# 08 — MLflow relog clean

Objectif :
- relire les artefacts déjà sauvegardés par `06_modeling.ipynb` et `07_tuning_and_test_final.ipynb`
- recréer des runs MLflow propres
- séparer :
  - validation / comparaison modèles
  - tuning
  - test final
- éviter de réentraîner inutilement les modèles

In [1]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Racine projet
PROJECT_ROOT = Path("..").resolve()

# Dossiers existants (issus des notebooks 06 et 07)
ART_06 = PROJECT_ROOT / "artifacts" / "06_modeling"
ART_06_TBL = ART_06 / "tables"
ART_06_MET = ART_06 / "metrics"
ART_06_PRD = ART_06 / "predictions"
ART_06_PLT = ART_06 / "plots"
ART_06_SUM = ART_06 / "summary"

ART_07 = PROJECT_ROOT / "artifacts" / "07_tuning_test"
ART_07_TBL = ART_07 / "tables"
ART_07_MET = ART_07 / "metrics"
ART_07_PRD = ART_07 / "predictions"
ART_07_PLT = ART_07 / "plots"
ART_07_SUM = ART_07 / "summary"

# Nouveau dossier pour le notebook 08
ART_08 = PROJECT_ROOT / "artifacts" / "08_mlflow_relog_clean"
ART_08_TBL = ART_08 / "tables"
ART_08_PLT = ART_08 / "plots"
ART_08_SUM = ART_08 / "summary"

for d in [ART_08, ART_08_TBL, ART_08_PLT, ART_08_SUM]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("ART_06 exists:", ART_06.exists())
print("ART_07 exists:", ART_07.exists())
print("ART_08       :", ART_08)

PROJECT_ROOT : C:\Users\El-fahad COMBO\Desktop\Serie_temporelle
ART_06 exists: True
ART_07 exists: True
ART_08       : C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\08_mlflow_relog_clean


In [2]:
import mlflow
from mlflow.tracking import MlflowClient

MLFLOW_DIR = PROJECT_ROOT / "mlflow"
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = (MLFLOW_DIR / "mlflow.db").resolve()
MLRUN_DIR = (MLFLOW_DIR / "mlrun").resolve()
MLRUN_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT = MLRUN_DIR.as_uri()

mlflow.set_tracking_uri(f"sqlite:///{DB_PATH.as_posix()}")
client = MlflowClient()

EXPERIMENTS = {
    "modeling": "river_temp_ts_modeling_clean",
    "tuning": "river_temp_ts_tuning_clean",
    "final": "river_temp_ts_final_clean",
}

def get_or_create_experiment(name: str):
    exp = client.get_experiment_by_name(name)
    if exp is None:
        exp_id = client.create_experiment(name, artifact_location=ARTIFACT_ROOT)
        exp = client.get_experiment(exp_id)
    return exp

for exp_name in EXPERIMENTS.values():
    exp = get_or_create_experiment(exp_name)
    print(f"[OK] {exp.name} | id={exp.experiment_id} | lifecycle={exp.lifecycle_stage}")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

[OK] river_temp_ts_modeling_clean | id=48 | lifecycle=active
[OK] river_temp_ts_tuning_clean | id=49 | lifecycle=active
[OK] river_temp_ts_final_clean | id=50 | lifecycle=active
MLflow tracking URI: sqlite:///C:/Users/El-fahad COMBO/Desktop/Serie_temporelle/Mlflow/mlflow.db


In [8]:
def read_csv_if_exists(path: Path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"[MISSING] {path}")
        return None
    df = pd.read_csv(path, **kwargs)
    print(f"[OK] {path.name} -> {df.shape}")
    return df

def read_json_if_exists(path: Path):
    path = Path(path)
    if not path.exists():
        print(f"[MISSING] {path}")
        return None
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    print(f"[OK] {path.name}")
    return obj

def canon_model(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if "baseline" in s:
        return "baseline"
    if "sarimax" in s:
        return "sarimax"
    if "sarima" in s:
        return "sarima"
    if "arima" in s:
        return "arima"
    if "ridge" in s:
        return "ridge"
    if "ets" in s:
        return "ets"
    if "lasso" in s:
        return "lasso"
    if "rf" in s or "random_forest" in s:
        return "rf"
    return s.replace(" ", "_")

def canon_horizon(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"h2", "2h", "2_h"} or "h2" in s:
        return "h2"
    if s in {"d1", "1d", "24h", "d_1"} or "d1" in s:
        return "d1"
    if s in {"w1", "1w", "7d", "week"} or "w1" in s:
        return "w1"
    return s

def standardize_metric_table(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()
    out.columns = [str(c).strip().lower() for c in out.columns]

    rename_map = {
        "model_family": "model",
        "model_name": "model",
        "family": "model",
        "method": "model",
        "algo": "model",
        "metric_mae": "mae",
        "metric_rmse": "rmse",
        "metric_smape": "smape",
        "metric_r2": "r2",
        "n_predictions": "n_preds",
        "npreds": "n_preds",
        "mode_name": "mode",
    }
    out = out.rename(columns={c: rename_map[c] for c in out.columns if c in rename_map})

    for c in ["model", "horizon", "mode"]:
        if c not in out.columns:
            out[c] = np.nan

    out["model"] = out["model"].map(canon_model)
    out["horizon"] = out["horizon"].map(canon_horizon)

    if out["mode"].isna().all():
        out["mode"] = "expanding"

    for c in ["mae", "rmse", "smape", "r2", "n_preds"]:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors="coerce")

    keep = ["model", "horizon", "mode", "mae", "rmse", "smape", "r2", "n_preds"]
    extras = [c for c in ["alpha", "order", "seasonal_order", "exog", "n_folds_used"] if c in out.columns]

    out = out[keep + extras].copy()
    out = out.dropna(subset=["model", "horizon"], how="any")
    return out

def save_fig(fig, filename: str) -> Path:
    path = ART_08_PLT / filename
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return path

def write_single_row_csv(row: pd.Series, filename: str) -> Path:
    path = ART_08_TBL / filename
    pd.DataFrame([row]).to_csv(path, index=False)
    return path

def log_file_if_exists(path, artifact_path: str = None):
    if path is None:
        return

    path = Path(path)
    if path.exists():
        if artifact_path is None:
            mlflow.log_artifact(str(path))
        else:
            mlflow.log_artifact(str(path), artifact_path=artifact_path)

def set_experiment_by_name(name: str):
    exp = get_or_create_experiment(name)
    mlflow.set_experiment(experiment_id=exp.experiment_id)
    return exp

In [21]:

MODEL_ORDER = ["baseline", "naive_last", "snaive_d", "ets", "arima", "sarima", "sarimax", "ridge", "lasso", "rf"]

def reorder_index(df, order=MODEL_ORDER):
    existing = [x for x in order if x in df.index]
    others = [x for x in df.index if x not in existing]
    return df.loc[existing + others]

def format_model_label(x: str) -> str:
    x = str(x)
    mapping = {
        "naive_last": "Naive last",
        "snaive_d": "Seasonal naive",
        "ets": "ETS",
        "arima": "ARIMA",
        "sarima": "SARIMA",
        "sarimax": "SARIMAX",
        "ridge": "Ridge",
        "baseline": "Baseline",
        "lasso": "Lasso",
        "rf": "Random Forest",
    }
    return mapping.get(x, x.replace("_", " ").title())

def format_horizon_label(x: str) -> str:
    mapping = {
        "h2": "H2",
        "d1": "D1",
        "w1": "W1",
    }
    return mapping.get(str(x), str(x).upper())

def annotate_bars(ax, decimals=3, fontsize=9, min_height=1e-12):
    """
    Ajoute les labels au-dessus des barres avec un offset propre,
    sans coller à la barre ni sortir du graphe.
    """
    ymax = ax.get_ylim()[1]
    offset = ymax * 0.015

    for container in ax.containers:
        labels = []
        for bar in container:
            h = bar.get_height()
            if pd.isna(h) or abs(h) <= min_height:
                labels.append("")
            else:
                labels.append(f"{h:.{decimals}f}")
        ax.bar_label(
            container,
            labels=labels,
            padding=3,
            fontsize=fontsize
        )

def apply_clean_axis_style(ax, title, ylabel, xlabel=""):
    ax.set_title(title, pad=14, fontsize=14, fontweight="semibold")
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.grid(True, axis="y", alpha=0.22)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def set_top_margin_for_labels(ax, frac=0.12):
    ymin, ymax = ax.get_ylim()
    if ymax > 0:
        ax.set_ylim(ymin, ymax * (1 + frac))

### Charger et reconstruire la table propre de comparaison VALIDATION

In [4]:
m06_all = read_csv_if_exists(ART_06_MET / "all_models_metrics_concat.csv")

m06_baseline = read_csv_if_exists(ART_06_TBL / "baseline_best_by_mode_horizon.csv")
m06_ets = read_csv_if_exists(ART_06_MET / "ets_metrics_global.csv")
m06_arima = read_csv_if_exists(ART_06_MET / "arima_metrics_folds.csv")
m06_sarima = read_csv_if_exists(ART_06_MET / "sarima_metrics_folds.csv")
m06_sarimax = read_csv_if_exists(ART_06_MET / "sarimax_metrics_folds.csv")
m06_ridge = read_csv_if_exists(ART_06_MET / "ridge_metrics_folds.csv")

parts = []

if m06_all is not None and not m06_all.empty:
    tmp = standardize_metric_table(m06_all)
    if not tmp.empty:
        parts.append(tmp)

fallbacks = [
    (m06_baseline, "baseline"),
    (m06_ets, "ets"),
    (m06_arima, "arima"),
    (m06_sarima, "sarima"),
    (m06_sarimax, "sarimax"),
    (m06_ridge, "ridge"),
]

for df_raw, default_model in fallbacks:
    if df_raw is None or df_raw.empty:
        continue
    tmp = standardize_metric_table(df_raw)
    if tmp.empty:
        continue
    tmp["model"] = tmp["model"].fillna(default_model)
    parts.append(tmp)

if len(parts) == 0:
    raise ValueError("Aucune table de métriques exploitable trouvée dans artifacts/06_modeling.")

modeling_df = pd.concat(parts, ignore_index=True)

if modeling_df["mode"].astype(str).str.contains("expand", case=False, na=False).any():
    modeling_df = modeling_df[modeling_df["mode"].astype(str).str.contains("expand", case=False, na=False)].copy()

# une ligne finale par (model, horizon)
modeling_df = (
    modeling_df
    .sort_values(["horizon", "mae", "rmse", "smape"], na_position="last")
    .drop_duplicates(subset=["model", "horizon"], keep="first")
    .reset_index(drop=True))

baseline_mae_map = (
    modeling_df.query("model == 'baseline'")
    .dropna(subset=["horizon", "mae"])
    .set_index("horizon")["mae"]
    .to_dict())

def skill_vs_baseline(row):
    b = baseline_mae_map.get(row["horizon"])
    if pd.isna(row["mae"]) or b is None or pd.isna(b) or b == 0:
        return np.nan
    return 1 - (row["mae"] / b)

modeling_df["skill_vs_baseline_mae"] = modeling_df.apply(skill_vs_baseline, axis=1)

leaderboard_val = modeling_df.sort_values(["horizon", "mae"], na_position="last").reset_index(drop=True)
leaderboard_val.to_csv(ART_08_TBL / "leaderboard_modeling_val_clean.csv", index=False)

display(leaderboard_val)
print("Saved:", ART_08_TBL / "leaderboard_modeling_val_clean.csv")

[OK] all_models_metrics_concat.csv -> (18, 8)
[OK] baseline_best_by_mode_horizon.csv -> (3, 7)
[OK] ets_metrics_global.csv -> (3, 8)
[MISSING] C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\06_modeling\metrics\arima_metrics_folds.csv
[MISSING] C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\06_modeling\metrics\sarima_metrics_folds.csv
[MISSING] C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\06_modeling\metrics\sarimax_metrics_folds.csv
[MISSING] C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\06_modeling\metrics\ridge_metrics_folds.csv


,model,horizon,mode,mae,rmse,smape,r2,n_preds,skill_vs_baseline_mae
0,naive_last,d1,expanding,0.417690,0.559507,4.331587,NaN,84,NaN
1,sarimax,d1,expanding,0.451987,0.573446,4.464331,0.983526,84,NaN
2,ridge,d1,expanding,0.460987,0.554822,4.868392,0.984579,84,NaN
3,sarima,d1,expanding,0.494784,0.618926,4.865205,0.980810,84,NaN
4,arima,d1,expanding,0.605947,0.841778,5.056943,0.964502,84,NaN
5,ets,d1,expanding,0.666001,0.871499,6.316652,0.961951,84,NaN
6,ridge,h2,expanding,0.074631,0.103531,0.645719,0.999514,84,NaN
7,sarima,h2,expanding,0.082907,0.115454,0.707903,0.999396,84,NaN
8,sarimax,h2,expanding,0.083805,0.115466,0.713602,0.999396,84,NaN
9,arima,h2,expanding,0.086851,0.122118,0.750692,0.999324,84,NaN


Saved: C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\08_mlflow_relog_clean\tables\leaderboard_modeling_val_clean.csv


In [ ]:
# =========================
# Chart 1 : Validation MAE
# =========================
plot_df = leaderboard_val.pivot(index="model", columns="horizon", values="mae")

PLOT_VAL_MAE = None
PLOT_VAL_SKILL = None

if not plot_df.empty:
    plot_df = reorder_index(plot_df)
    plot_df.index = [format_model_label(x) for x in plot_df.index]
    plot_df.columns = [format_horizon_label(x) for x in plot_df.columns]

    fig, ax = plt.subplots(figsize=(12, 5.8))
    plot_df.plot(kind="bar", ax=ax, width=0.78)

    apply_clean_axis_style(
        ax,
        title="Validation — comparaison des modèles par horizon (MAE)",
        ylabel="MAE",
        xlabel="Modèle"
    )

    ax.tick_params(axis="x", rotation=25, labelsize=10)
    ax.legend(title="Horizon", frameon=False, ncol=len(plot_df.columns), loc="upper right")

    set_top_margin_for_labels(ax, frac=0.14)
    annotate_bars(ax, decimals=3, fontsize=9)

    PLOT_VAL_MAE = save_fig(fig, "validation_compare_mae_by_horizon.png")
    print("Saved:", PLOT_VAL_MAE)

# ====================================
# Chart 2 : Validation skill baseline
# ====================================
skill_df = leaderboard_val.dropna(subset=["skill_vs_baseline_mae"]).copy()

if not skill_df.empty:
    skill_pivot = skill_df.pivot(index="model", columns="horizon", values="skill_vs_baseline_mae")
    skill_pivot = reorder_index(skill_pivot)
    skill_pivot.index = [format_model_label(x) for x in skill_pivot.index]
    skill_pivot.columns = [format_horizon_label(x) for x in skill_pivot.columns]

    fig, ax = plt.subplots(figsize=(12, 5.8))
    skill_pivot.plot(kind="bar", ax=ax, width=0.78)

    apply_clean_axis_style(
        ax,
        title="Validation — skill score vs baseline (MAE)",
        ylabel="Skill score",
        xlabel="Modèle"
    )

    ax.axhline(0.0, linestyle="--", linewidth=1)
    ax.tick_params(axis="x", rotation=25, labelsize=10)
    ax.legend(title="Horizon", frameon=False, ncol=len(skill_pivot.columns), loc="upper right")

    set_top_margin_for_labels(ax, frac=0.18)
    annotate_bars(ax, decimals=3, fontsize=9)

    PLOT_VAL_SKILL = save_fig(fig, "validation_skill_vs_baseline.png")
    print("Saved:", PLOT_VAL_SKILL)
else:
    PLOT_VAL_SKILL = None

Saved: C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\08_mlflow_relog_clean\plots\validation_compare_mae_by_horizon.png


In [22]:
set_experiment_by_name(EXPERIMENTS["modeling"])

source_files_06 = {
    "baseline": ART_06_TBL / "baseline_best_by_mode_horizon.csv",
    "ets": ART_06_MET / "ets_metrics_global.csv",
    "arima": ART_06_MET / "arima_metrics_folds.csv",
    "sarima": ART_06_MET / "sarima_metrics_folds.csv",
    "sarimax": ART_06_MET / "sarimax_metrics_folds.csv",
    "ridge": ART_06_MET / "ridge_metrics_folds.csv",
}

for _, row in leaderboard_val.iterrows():
    model = str(row["model"])
    horizon = str(row["horizon"])
    run_name = f"{model}_{horizon}_val_clean"

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("source_notebook", "06_modeling")
        mlflow.set_tag("stage", "modeling")
        mlflow.set_tag("split", "val")
        mlflow.set_tag("protocol", "fold_origin")
        mlflow.set_tag("mode", str(row.get("mode", "expanding")))
        mlflow.set_tag("model_family", model)
        mlflow.set_tag("horizon", horizon)
        mlflow.set_tag("relogged", "yes")

        mlflow.log_metric("val_mae", float(row["mae"]))
        if pd.notna(row["rmse"]):
            mlflow.log_metric("val_rmse", float(row["rmse"]))
        if pd.notna(row["smape"]):
            mlflow.log_metric("val_smape", float(row["smape"]))
        if pd.notna(row["r2"]):
            mlflow.log_metric("val_r2", float(row["r2"]))
        if pd.notna(row["n_preds"]):
            mlflow.log_metric("n_preds", float(row["n_preds"]))
        if pd.notna(row["skill_vs_baseline_mae"]):
            mlflow.log_metric("skill_vs_baseline_mae", float(row["skill_vs_baseline_mae"]))

        row_csv = write_single_row_csv(row, f"{run_name}.csv")
        mlflow.log_artifact(str(row_csv), artifact_path="tables")

        log_file_if_exists(ART_08_TBL / "leaderboard_modeling_val_clean.csv", artifact_path="tables")
        log_file_if_exists(ART_06_MET / "all_models_metrics_concat.csv", artifact_path="source_metrics")
        log_file_if_exists(source_files_06.get(model), artifact_path="source_metrics")
        log_file_if_exists(PLOT_VAL_MAE, artifact_path="plots")
        if PLOT_VAL_SKILL is not None:
            log_file_if_exists(PLOT_VAL_SKILL, artifact_path="plots")

print("OK - relog validation clean terminé.")

OK - relog validation clean terminé.


### Charger et construire le résumé TUNING (before / after)

In [10]:
# fichiers tuning
ridge_grid = read_csv_if_exists(ART_07_TBL / "tuning_ridge_h2_grid.csv")
sarimax_fast = read_csv_if_exists(ART_07_TBL / "tuning_sarimax_d1_fast_screening.csv")
sarimax_top = read_csv_if_exists(ART_07_TBL / "tuning_sarimax_d1_top2_full.csv")
selected_final_json = read_json_if_exists(ART_07_SUM / "selected_models_final.json")

if ridge_grid is None or ridge_grid.empty:
    raise ValueError("tuning_ridge_h2_grid.csv manquant ou vide.")
if sarimax_top is None or sarimax_top.empty:
    raise ValueError("tuning_sarimax_d1_top2_full.csv manquant ou vide.")

ridge_grid_std = standardize_metric_table(ridge_grid)
sarimax_top_std = standardize_metric_table(sarimax_top)

ridge_before = leaderboard_val.query("model == 'ridge' and horizon == 'h2'").sort_values("mae").iloc[0]
ridge_after = ridge_grid_std.sort_values("mae").iloc[0]

sarimax_before = leaderboard_val.query("model == 'sarimax' and horizon == 'd1'").sort_values("mae").iloc[0]
sarimax_after = sarimax_top_std.sort_values("mae").iloc[0]

tuning_summary = pd.DataFrame([
    {
        "model": "ridge",
        "horizon": "h2",
        "mae_before": float(ridge_before["mae"]),
        "mae_after": float(ridge_after["mae"]),
        "delta_mae": float(ridge_before["mae"] - ridge_after["mae"]),
        "rmse_before": float(ridge_before["rmse"]) if pd.notna(ridge_before["rmse"]) else np.nan,
        "rmse_after": float(ridge_after["rmse"]) if pd.notna(ridge_after["rmse"]) else np.nan,
        "smape_before": float(ridge_before["smape"]) if pd.notna(ridge_before["smape"]) else np.nan,
        "smape_after": float(ridge_after["smape"]) if pd.notna(ridge_after["smape"]) else np.nan,
        "best_alpha": float(ridge_grid.sort_values("mae").iloc[0]["alpha"]),
    },
    {
        "model": "sarimax",
        "horizon": "d1",
        "mae_before": float(sarimax_before["mae"]),
        "mae_after": float(sarimax_after["mae"]),
        "delta_mae": float(sarimax_before["mae"] - sarimax_after["mae"]),
        "rmse_before": float(sarimax_before["rmse"]) if pd.notna(sarimax_before["rmse"]) else np.nan,
        "rmse_after": float(sarimax_after["rmse"]) if pd.notna(sarimax_after["rmse"]) else np.nan,
        "smape_before": float(sarimax_before["smape"]) if pd.notna(sarimax_before["smape"]) else np.nan,
        "smape_after": float(sarimax_after["smape"]) if pd.notna(sarimax_after["smape"]) else np.nan,
        "best_order": str(sarimax_top.sort_values("mae").iloc[0]["order"]),
        "best_seasonal_order": str(sarimax_top.sort_values("mae").iloc[0]["seasonal_order"]),
    },
])

tuning_summary.to_csv(ART_08_TBL / "tuning_summary_clean.csv", index=False)
display(tuning_summary)
print("Saved:", ART_08_TBL / "tuning_summary_clean.csv")

[OK] tuning_ridge_h2_grid.csv -> (7, 9)
[OK] tuning_sarimax_d1_fast_screening.csv -> (4, 12)
[OK] tuning_sarimax_d1_top2_full.csv -> (2, 10)
[OK] selected_models_final.json


,model,horizon,mae_before,mae_after,delta_mae,rmse_before,rmse_after,smape_before,smape_after,best_alpha,best_order,best_seasonal_order
0,ridge,h2,0.074631,0.074358,0.000274,0.103531,0.103252,0.645719,0.642278,10.0,NaN,NaN
1,sarimax,d1,0.451987,0.435637,0.016350,0.573446,0.605570,4.464331,4.208795,NaN,"(2, 0, 1)","(1, 0, 1, 12)"


Saved: C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\08_mlflow_relog_clean\tables\tuning_summary_clean.csv


In [23]:
# Chart before / after tuning
plot_df = tuning_summary.copy()
plot_df["label"] = plot_df["model"].map(format_model_label) + " (" + plot_df["horizon"].map(format_horizon_label) + ")"
plot_df = plot_df.set_index("label")[["mae_before", "mae_after"]]
plot_df = plot_df.rename(columns={"mae_before": "Avant tuning", "mae_after": "Après tuning"})

fig, ax = plt.subplots(figsize=(9.5, 5.5))
plot_df.plot(kind="bar", ax=ax, width=0.72)

apply_clean_axis_style(
    ax,
    title="Tuning — MAE avant / après",
    ylabel="MAE",
    xlabel="Modèle / horizon"
)

ax.tick_params(axis="x", rotation=20, labelsize=11)
ax.legend(frameon=False, ncol=2, loc="upper left")

set_top_margin_for_labels(ax, frac=0.12)
annotate_bars(ax, decimals=3, fontsize=10)

PLOT_TUNING = save_fig(fig, "tuning_before_after_mae.png")
print("Saved:", PLOT_TUNING)

# Relog MLflow
set_experiment_by_name(EXPERIMENTS["tuning"])

for _, row in tuning_summary.iterrows():
    model = row["model"]
    horizon = row["horizon"]
    run_name = f"tuning_{model}_{horizon}_clean"

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("source_notebook", "07_tuning_and_test_final")
        mlflow.set_tag("stage", "tuning")
        mlflow.set_tag("split", "val")
        mlflow.set_tag("model_family", model)
        mlflow.set_tag("horizon", horizon)
        mlflow.set_tag("relogged", "yes")

        mlflow.log_metric("before_val_mae", float(row["mae_before"]))
        mlflow.log_metric("after_val_mae", float(row["mae_after"]))
        mlflow.log_metric("delta_val_mae", float(row["delta_mae"]))

        if pd.notna(row["rmse_before"]):
            mlflow.log_metric("before_val_rmse", float(row["rmse_before"]))
        if pd.notna(row["rmse_after"]):
            mlflow.log_metric("after_val_rmse", float(row["rmse_after"]))
        if pd.notna(row["smape_before"]):
            mlflow.log_metric("before_val_smape", float(row["smape_before"]))
        if pd.notna(row["smape_after"]):
            mlflow.log_metric("after_val_smape", float(row["smape_after"]))

        if model == "ridge" and "best_alpha" in row and pd.notna(row["best_alpha"]):
            mlflow.log_param("best_alpha", float(row["best_alpha"]))
            log_file_if_exists(ART_07_TBL / "tuning_ridge_h2_grid.csv", artifact_path="tables")
            log_file_if_exists(ART_07_PRD / "ridge_val_preds_best_alpha.csv", artifact_path="predictions")

        if model == "sarimax":
            if "best_order" in row and pd.notna(row["best_order"]):
                mlflow.log_param("best_order", str(row["best_order"]))
            if "best_seasonal_order" in row and pd.notna(row["best_seasonal_order"]):
                mlflow.log_param("best_seasonal_order", str(row["best_seasonal_order"]))
            log_file_if_exists(ART_07_TBL / "tuning_sarimax_d1_fast_screening.csv", artifact_path="tables")
            log_file_if_exists(ART_07_TBL / "tuning_sarimax_d1_top2_full.csv", artifact_path="tables")

        row_csv = write_single_row_csv(row, f"{run_name}.csv")
        mlflow.log_artifact(str(row_csv), artifact_path="tables")
        log_file_if_exists(ART_08_TBL / "tuning_summary_clean.csv", artifact_path="tables")
        log_file_if_exists(PLOT_TUNING, artifact_path="plots")

print("OK - relog tuning clean terminé.")

Saved: C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\08_mlflow_relog_clean\plots\tuning_before_after_mae.png
OK - relog tuning clean terminé.


In [12]:
final_test_global = read_csv_if_exists(ART_07_MET / "final_test_metrics_global.csv")

# fallbacks si besoin
test_h2_ridge_global = read_csv_if_exists(ART_07_MET / "final_test_metrics_h2_ridge_global.csv")
test_d1_base_global = read_csv_if_exists(ART_07_MET / "final_test_metrics_d1_baseline_global.csv")
test_d1_smx_global = read_csv_if_exists(ART_07_MET / "final_test_metrics_d1_sarimax_global.csv")

parts_final = []

if final_test_global is not None and not final_test_global.empty:
    tmp = standardize_metric_table(final_test_global)
    if not tmp.empty:
        parts_final.append(tmp)

fallback_final = [
    (test_h2_ridge_global, "ridge", "h2"),
    (test_d1_base_global, "baseline", "d1"),
    (test_d1_smx_global, "sarimax", "d1"),
]

for df_raw, default_model, default_horizon in fallback_final:
    if df_raw is None or df_raw.empty:
        continue
    tmp = standardize_metric_table(df_raw)
    if tmp.empty:
        tmp = pd.DataFrame([{
            "model": default_model,
            "horizon": default_horizon,
            "mode": "walk_forward_continuous",
            "mae": pd.to_numeric(df_raw.iloc[0].get("mae", np.nan), errors="coerce"),
            "rmse": pd.to_numeric(df_raw.iloc[0].get("rmse", np.nan), errors="coerce"),
            "smape": pd.to_numeric(df_raw.iloc[0].get("smape", np.nan), errors="coerce"),
            "r2": pd.to_numeric(df_raw.iloc[0].get("r2", np.nan), errors="coerce"),
            "n_preds": pd.to_numeric(df_raw.iloc[0].get("n_preds", np.nan), errors="coerce"),
        }])
    else:
        tmp["model"] = tmp["model"].fillna(default_model)
        tmp["horizon"] = tmp["horizon"].fillna(default_horizon)
    parts_final.append(tmp)

if len(parts_final) == 0:
    raise ValueError("Aucune table TEST exploitable trouvée dans artifacts/07_tuning_test.")

final_df = pd.concat(parts_final, ignore_index=True)
final_df = (
    final_df
    .sort_values(["horizon", "mae"], na_position="last")
    .drop_duplicates(subset=["model", "horizon"], keep="first")
    .reset_index(drop=True)
)

baseline_test_map = (
    final_df.query("model == 'baseline'")
    .dropna(subset=["horizon", "mae"])
    .set_index("horizon")["mae"]
    .to_dict()
)

def test_skill_vs_baseline(row):
    b = baseline_test_map.get(row["horizon"])
    if pd.isna(row["mae"]) or b is None or pd.isna(b) or b == 0:
        return np.nan
    return 1 - (row["mae"] / b)

final_df["skill_vs_baseline_mae"] = final_df.apply(test_skill_vs_baseline, axis=1)
final_df.to_csv(ART_08_TBL / "final_test_clean.csv", index=False)

display(final_df)
print("Saved:", ART_08_TBL / "final_test_clean.csv")

[OK] final_test_metrics_global.csv -> (3, 7)
[OK] final_test_metrics_h2_ridge_global.csv -> (1, 5)
[OK] final_test_metrics_d1_baseline_global.csv -> (1, 5)
[OK] final_test_metrics_d1_sarimax_global.csv -> (1, 5)


,model,horizon,mode,mae,rmse,smape,r2,n_preds,skill_vs_baseline_mae
0,baseline,d1,expanding,0.528918,0.679535,3.264550,0.953943,8015.0,0.000000
1,sarimax,d1,expanding,0.603093,0.776109,3.689039,0.939922,8015.0,-0.140239
2,ridge,h2,expanding,0.256432,0.324603,1.540712,0.989509,8092.0,NaN


Saved: C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\08_mlflow_relog_clean\tables\final_test_clean.csv


In [ ]:
plot_df = final_df.pivot(index="model", columns="horizon", values="mae")

PLOT_TEST_MAE = None

if not plot_df.empty:
    plot_df = reorder_index(plot_df)
    plot_df.index = [format_model_label(x) for x in plot_df.index]
    plot_df.columns = [format_horizon_label(x) for x in plot_df.columns]

    fig, ax = plt.subplots(figsize=(9.5, 5.5))
    plot_df.plot(kind="bar", ax=ax, width=0.72)

    apply_clean_axis_style(
        ax,
        title="TEST final — comparaison des modèles par horizon (MAE)",
        ylabel="MAE",
        xlabel="Modèle"
    )

    ax.tick_params(axis="x", rotation=20, labelsize=11)
    ax.legend(title="Horizon", frameon=False, ncol=len(plot_df.columns), loc="upper right")

    set_top_margin_for_labels(ax, frac=0.14)
    annotate_bars(ax, decimals=3, fontsize=10)

    PLOT_TEST_MAE = save_fig(fig, "final_test_compare_mae_by_horizon.png")
    print("Saved:", PLOT_TEST_MAE)
else:
    print("Aucun graphique TEST généré : plot_df vide.")

Saved: C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\08_mlflow_relog_clean\plots\final_test_compare_mae_by_horizon.png


: 

In [16]:
def get_final_pred_file(model: str, horizon: str):
    model = canon_model(model)
    horizon = canon_horizon(horizon)

    if model == "ridge" and horizon == "h2":
        return ART_07_PRD / "preds_test_h2_ridge.csv"
    if model == "baseline" and horizon == "d1":
        return ART_07_PRD / "preds_test_d1_baseline_daily.csv"
    if model == "sarimax" and horizon == "d1":
        return ART_07_PRD / "preds_test_d1_sarimax_tuned.csv"
    return None

def get_final_station_metric_file(model: str, horizon: str):
    model = canon_model(model)
    horizon = canon_horizon(horizon)

    if model == "ridge" and horizon == "h2":
        return ART_07_MET / "final_test_metrics_h2_ridge_by_station.csv"
    if model == "baseline" and horizon == "d1":
        return ART_07_MET / "final_test_metrics_d1_baseline_by_station.csv"
    if model == "sarimax" and horizon == "d1":
        return ART_07_MET / "final_test_metrics_d1_sarimax_by_station.csv"
    return None

FIG_07 = PROJECT_ROOT / "reports" / "figures" / "07_tuning_test"

set_experiment_by_name(EXPERIMENTS["final"])

# 1) un run par modèle-horizon final
for _, row in final_df.iterrows():
    model = str(row["model"])
    horizon = str(row["horizon"])
    run_name = f"{model}_{horizon}_test_clean"

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("source_notebook", "07_tuning_and_test_final")
        mlflow.set_tag("stage", "test_final")
        mlflow.set_tag("split", "test")
        mlflow.set_tag("protocol", "walk_forward_continuous")
        mlflow.set_tag("model_family", model)
        mlflow.set_tag("horizon", horizon)
        mlflow.set_tag("relogged", "yes")

        mlflow.log_metric("test_mae", float(row["mae"]))
        if pd.notna(row["rmse"]):
            mlflow.log_metric("test_rmse", float(row["rmse"]))
        if pd.notna(row["smape"]):
            mlflow.log_metric("test_smape", float(row["smape"]))
        if pd.notna(row["r2"]):
            mlflow.log_metric("test_r2", float(row["r2"]))
        if pd.notna(row["n_preds"]):
            mlflow.log_metric("n_preds", float(row["n_preds"]))
        if pd.notna(row["skill_vs_baseline_mae"]):
            mlflow.log_metric("skill_vs_baseline_mae", float(row["skill_vs_baseline_mae"]))

        row_csv = write_single_row_csv(row, f"{run_name}.csv")
        mlflow.log_artifact(str(row_csv), artifact_path="tables")
        log_file_if_exists(ART_08_TBL / "final_test_clean.csv", artifact_path="tables")
        log_file_if_exists(ART_07_MET / "final_test_metrics_global.csv", artifact_path="tables")
        log_file_if_exists(PLOT_TEST_MAE, artifact_path="plots")

        pred_file = get_final_pred_file(model, horizon)
        station_file = get_final_station_metric_file(model, horizon)
        log_file_if_exists(pred_file, artifact_path="predictions")
        log_file_if_exists(station_file, artifact_path="metrics_by_station")

        # figures déjà créées dans le notebook 07
        if model == "ridge" and horizon == "h2":
            log_file_if_exists(FIG_07 / "test_ridge_h2_station_817.png", artifact_path="report_figures")
            log_file_if_exists(FIG_07 / "test_scatter_ridge_h2.png", artifact_path="report_figures")
        if model == "sarimax" and horizon == "d1":
            log_file_if_exists(FIG_07 / "test_compare_d1_station_817.png", artifact_path="report_figures")

# 2) run résumé global
with mlflow.start_run(run_name="final_summary_clean"):
    mlflow.set_tag("source_notebook", "07_tuning_and_test_final")
    mlflow.set_tag("stage", "summary")
    mlflow.set_tag("relogged", "yes")

    # métriques résumé : meilleure MAE par horizon
    for horizon in sorted(final_df["horizon"].dropna().unique()):
        sub = final_df[final_df["horizon"] == horizon].sort_values("mae")
        if len(sub) > 0:
            best = sub.iloc[0]
            mlflow.log_metric(f"best_{horizon}_test_mae", float(best["mae"]))

    log_file_if_exists(ART_08_TBL / "final_test_clean.csv", artifact_path="tables")
    log_file_if_exists(ART_07_SUM / "selected_models_final.json", artifact_path="summary")
    log_file_if_exists(PLOT_TEST_MAE, artifact_path="plots")

print("OK - relog final clean terminé.")

OK - relog final clean terminé.


In [17]:
print("=== Fichiers 08 générés ===")
for p in sorted(ART_08.rglob("*")):
    if p.is_file():
        print(p.relative_to(PROJECT_ROOT))

=== Fichiers 08 générés ===
artifacts\08_mlflow_relog_clean\plots\final_test_compare_mae_by_horizon.png
artifacts\08_mlflow_relog_clean\plots\tuning_before_after_mae.png
artifacts\08_mlflow_relog_clean\plots\validation_compare_mae_by_horizon.png
artifacts\08_mlflow_relog_clean\tables\arima_d1_val_clean.csv
artifacts\08_mlflow_relog_clean\tables\arima_h2_val_clean.csv
artifacts\08_mlflow_relog_clean\tables\arima_w1_val_clean.csv
artifacts\08_mlflow_relog_clean\tables\baseline_d1_test_clean.csv
artifacts\08_mlflow_relog_clean\tables\ets_d1_val_clean.csv
artifacts\08_mlflow_relog_clean\tables\ets_h2_val_clean.csv
artifacts\08_mlflow_relog_clean\tables\ets_w1_val_clean.csv
artifacts\08_mlflow_relog_clean\tables\final_test_clean.csv
artifacts\08_mlflow_relog_clean\tables\leaderboard_modeling_val_clean.csv
artifacts\08_mlflow_relog_clean\tables\naive_last_d1_val_clean.csv
artifacts\08_mlflow_relog_clean\tables\naive_last_h2_val_clean.csv
artifacts\08_mlflow_relog_clean\tables\ridge_d1_val_cl

## Bonus — Enregistrer le modèle final dans MLflow Models

Objectif :
- logguer un vrai modèle MLflow
- créer/mettre à jour une entrée dans l’onglet `Models`
- commencer par le modèle final `Ridge h2`